# Sampling with Langevin dynamics on two bumps

The deck *Sampling with Langevin dynamics* ends on two algorithms. Algorithm 3 trains one network $s_\theta(x,\sigma)$ on the score of noisy data at many levels of noise, and Algorithm 4, annealed Langevin sampling, turns that network into samples. This notebook runs both on the example of pages 18 to 21 of the deck, a pdf on the line with two bumps:

$$p^\star=0.2\,\mathcal N(-3, 0.5^2) + 0.8\,\mathcal N(3, 0.5^2).$$

* We know $p^\star$ here, so we can compare every result with the truth. Algorithms 2 to 4 never use it: they see only the samples $x_1,\dots,x_n$.
* Sections 2 to 5 follow the deck: Langevin steps with the exact score (page 10) and why they fail on two bumps; NCSM at one level (Algorithm 2) and why one level is too few; one network for all the levels (Algorithm 3); annealed Langevin sampling (Algorithm 4).
* Section 6 reads the network as a diffusion model does (pages 26 and 27).

The names follow the deck:

| The deck | This notebook |
|---|---|
| $p^\star$ and $p_\sigma$ | `pdf(x)` and `pdf(x, sigma)` |
| the samples $x_1,\dots,x_n$ | `data` |
| the score $\nabla_x\log p_\sigma(x)$ | `exact_score(x, sigma)` |
| the network $s_\theta(x,\sigma)$ | `s_theta`, a `ScoreNetwork` |
| $\varepsilon\sim\mathcal N(0,\sigma^2)$ and $\tilde x=x+\varepsilon$ | `eps` and `x_tilde` |
| the loss $\hat J(\theta)$ of a minibatch of $B$ samples | `J_hat` and `B` |
| the levels $\sigma_1>\dots>\sigma_K$ | `sigmas` |
| $\Delta$, $\Delta_k$ and $T_k$ | `Delta`, `Delta_k` and `steps_per_level` |
| $\varepsilon_\theta=-\sigma\,s_\theta$ | `eps_theta(x, sigma)` |

Run the cells in order, or choose Runtime, Run all. Everything runs on the CPU in well under a minute.

In [ ]:
#@title Imports { display-mode: "form" }
import math
import torch
from torch import nn
import matplotlib.pyplot as plt

## 1. Two bumps

A point of $p^\star$ comes from the left bump with chance $0.2$ and from the right bump with chance $0.8$; each bump is a Gaussian of width $0.5$. `sample_p_star(n)` draws $n$ points of $p^\star$. `pdf(x, sigma)` is the pdf of $p_\sigma$, the pdf of $X+\varepsilon$ with $X\sim p^\star$ and $\varepsilon\sim\mathcal N(0,\sigma^2)$: the noise widens each bump from $0.5^2$ to $0.5^2+\sigma^2$, so $p_\sigma$ is again a mixture of two bumps, with the same weights.

`data` holds 5000 samples of $p^\star$. Sections 3 to 5 learn from these samples alone.

In [ ]:
torch.manual_seed(0)                                         # seeded: the same numbers on every run

WEIGHTS = [0.2, 0.8]                                         # the chance of each bump
MEANS = [-3.0, 3.0]                                          # the centre of each bump
WIDTH = 0.5                                                  # the width of each bump


def sample_p_star(n):
    # n points of p*: for each point a bump, picked with the chances WEIGHTS, then a point of that bump
    bump = torch.multinomial(torch.tensor(WEIGHTS), n, replacement=True)
    return torch.tensor(MEANS)[bump] + WIDTH * torch.randn(n)


def pdf(x, sigma=0.0):
    # the pdf of p_sigma, the pdf of X + eps with X ~ p* and eps ~ N(0, sigma^2); sigma = 0 gives p*
    var = WIDTH ** 2 + sigma ** 2                            # the noise widens each bump
    return sum(w * torch.exp(-(x - m) ** 2 / (2 * var)) / math.sqrt(2 * math.pi * var)
               for w, m in zip(WEIGHTS, MEANS))


def shares(X):
    # the share of the points X nearest to each centre, in per cent
    nearest = (X[:, None] - torch.tensor(MEANS)).abs().argmin(dim=1)
    return [100 * (nearest == i).float().mean().item() for i in range(len(MEANS))]


data = sample_p_star(5000)                                   # x_1, ..., x_n
print("shares of the bumps in the data:", " and ".join(f"{s:.1f} %" for s in shares(data)))

In [ ]:
#@title Plot the data against p* (double-click to view the code) { display-mode: "form" }
xs = torch.linspace(-8, 8, 801)
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(data.numpy(), bins=96, range=(-8, 8), density=True, color="C0", alpha=0.45, label="the 5000 samples")
ax.plot(xs.numpy(), pdf(xs).numpy(), color="black", label="$p^\\star$")
ax.plot(xs.numpy(), pdf(xs, 1.0).numpy(), color="C1", linestyle="--", label="$p_\\sigma$, $\\sigma=1$")
ax.set(xlabel="$x$", ylabel="pdf")
ax.legend(loc="upper left")
plt.show()

## 2. Langevin steps with the exact score

Page 10 of the deck: with $f=\frac12\nabla_x\log p^\star$, the steps

$$X(t+\Delta)=X(t)+\tfrac12\,\nabla_x\log p^\star\bigl(X(t)\bigr)\,\Delta+\sqrt{\Delta}\,\varepsilon_t,\qquad\varepsilon_t\sim\mathcal N(0,1),$$

end at $p^\star$ for a large $T$. Here we know $p^\star$, so we can compute its score, and the score of every $p_\sigma$. For bumps $\mathcal N(m_i,v)$ with weights $w_i$,

$$\nabla_x\log p_\sigma(x)=\sum_i r_i(x)\,\frac{m_i-x}{v},\qquad v=0.5^2+\sigma^2,$$

where $r_i(x)$ is the chance that a point at $x$ came from bump $i$. Each bump pulls $x$ toward its centre, as strongly as it owns $x$. `torch.softmax` computes the $r_i(x)$ from the numbers $\log w_i-(x-m_i)^2/(2v)$, which keeps them accurate far from both bumps.

In [ ]:
def exact_score(x, sigma=0.0):
    # the score of p_sigma, d/dx log p_sigma(x), from the formula above
    var = WIDTH ** 2 + sigma ** 2
    logits = torch.stack([math.log(w) - (x - m) ** 2 / (2 * var) for w, m in zip(WEIGHTS, MEANS)])
    r = torch.softmax(logits, dim=0)                         # r[i]: the chance that x came from bump i
    return sum(r[i] * (MEANS[i] - x) / var for i in range(len(MEANS)))

`langevin_steps` runs the steps of page 10 for 2000 runs at once, each run a column of `path`. The runs start at $X(0)\sim\mathcal N(0,1)$, as on page 2, and take 1000 steps of $\Delta=0.005$.

In [ ]:
def langevin_steps(score, X, Delta, steps):
    # the steps of page 10 for every run at once; returns the path, one row per step, one column per run
    path = [X]
    for t in range(steps):
        eps = torch.randn(len(X))
        X = X + 0.5 * score(X) * Delta + math.sqrt(Delta) * eps
        path.append(X)
    return torch.stack(path)


X0 = torch.randn(2000)                                       # 2000 runs from N(0, 1)
path = langevin_steps(exact_score, X0, Delta=0.005, steps=1000)

In [ ]:
#@title `plot_runs` draws 30 runs and where all of them end (double-click to view the code) { display-mode: "form" }
def plot_runs(path, title):
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw={"width_ratios": [3, 2]})
    left.plot(path[:, :30].numpy(), color="C0", linewidth=0.5, alpha=0.6)
    left.set(xlabel="step", ylabel="$x$", ylim=(-10, 10), title=title)
    xs = torch.linspace(-6, 6, 601)
    right.hist(path[-1].numpy(), bins=72, range=(-6, 6), density=True, color="C0", alpha=0.45,
               label="where the runs end")
    right.plot(xs.numpy(), pdf(xs).numpy(), color="black", label="$p^\\star$")
    right.set(xlabel="$x$")
    right.set_ylim(bottom=0)
    right.legend(loc="upper left")
    plt.show()
    print("shares of the bumps at the end:", " and ".join(f"{s:.1f} %" for s in shares(path[-1])),
          "; p* gives", " and ".join(f"{100 * w:.0f} %" for w in WEIGHTS))


plot_runs(path, "Langevin steps with the exact score of $p^\\star$")

The two bumps have the right shape and the wrong weights: about half the runs end in the left bump, where $p^\star$ puts 20 %. Each run settles in the bump nearest its start, and $\mathcal N(0,1)$ puts about half the starts on each side. Between the bumps $p^\star$ is tiny, and a run would need a long series of kicks against the drift to cross. The next cell counts the runs that changed bumps after step 100.

In [ ]:
def nearest_bump(X):
    return (X[:, None] - torch.tensor(MEANS)).abs().argmin(dim=1)


changed = (nearest_bump(path[100]) != nearest_bump(path[-1])).sum().item()
print(changed, "of the 2000 runs changed bumps between step 100 and step 1000")

None of them did. For a large $T$ the runs would cross now and then, and the weights would come right; on two bumps this far apart, large means far more steps than this notebook can run. Page 21 of the deck shows the same with the score of $p_\sigma$ at a small $\sigma$.

## 3. Learning the score from the samples (Algorithm 2)

In practice $p^\star$ is unknown, and so is its score: all we have is `data`. NCSM (page 15) learns the score of the noisy data $\tilde x=x+\varepsilon$, $\varepsilon\sim\mathcal N(0,\sigma^2)$, from the data alone: the target of $s_\theta(\tilde x,\sigma)$ is $-\varepsilon/\sigma^2$.

The network $s_\theta(x,\sigma)$ reads $x$ and $\log\sigma$, has two hidden layers of 64 ReLUs and gives one number. That number is divided by $\sigma$, since the score of $p_\sigma$ is of the size $1/\sigma$: on page 22, $\sigma\,s_\theta$ is of the size of a $\mathcal N(0,1)$ number.

In [ ]:
class ScoreNetwork(nn.Module):
    # s_theta(x, sigma): x and log sigma in, two hidden layers of 64 ReLUs, one number out, divided by sigma
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, sigma):
        inputs = torch.stack([x, torch.full_like(x, math.log(sigma))], dim=1)   # one row per point
        return self.layers(inputs)[:, 0] / sigma

Algorithm 2 (page 17) trains it at one level. We take the smallest level of the deck, $\sigma=0.1$, where $p_\sigma$ is closest to $p^\star$. Every step takes a minibatch of $B=256$ samples, adds noise and moves the weights against the gradient of $\hat J(\theta)$, 2000 steps in all. Adam is gradient descent with a step size that it adapts for every weight.

In [ ]:
def train_one_level(sigma, steps=2000, B=256):
    # Algorithm 2 of page 17
    s_theta = ScoreNetwork()
    optimizer = torch.optim.Adam(s_theta.parameters(), lr=0.001)
    for step in range(steps):
        x = data[torch.randint(len(data), (B,))]               # a minibatch of the data
        eps = sigma * torch.randn(B)                           # eps ~ N(0, sigma^2)
        x_tilde = x + eps
        J_hat = ((s_theta(x_tilde, sigma) + eps / sigma ** 2) ** 2).sum() / (2 * B)
        optimizer.zero_grad()                                  # forget the gradient of the previous step
        J_hat.backward()                                       # the gradient of J_hat for every weight
        optimizer.step()                                       # move every weight
    return s_theta


s_theta_one_level = train_one_level(sigma=0.1)

In [ ]:
#@title `plot_scores` draws s_theta against the score, with noisy points under it (double-click to view the code) { display-mode: "form" }
def errors(network, sigma):
    # the mean distance between s_theta and the score: on the bumps (within 1 of a centre)
    # and between them (|x| < 1.5, the band of page 20)
    xs = torch.linspace(-6, 6, 1201)
    with torch.no_grad():
        gap = (network(xs, sigma) - exact_score(xs, sigma)).abs()
    on_bumps = (xs[:, None] - torch.tensor(MEANS)).abs().min(dim=1).values < 1
    return gap[on_bumps].mean().item(), gap[xs.abs() < 1.5].mean().item()


def plot_scores(network, levels):
    fig, axes = plt.subplots(1, len(levels), figsize=(4.2 * len(levels), 2.8), squeeze=False)
    xs = torch.linspace(-6, 6, 601)
    for ax, sigma in zip(axes[0], levels):
        with torch.no_grad():
            learned = network(xs, sigma)
        exact = exact_score(xs, sigma)
        top = 1.6 * exact.abs().max().item()                 # room for the legend above the curves
        noisy = sample_p_star(1000) + sigma * torch.randn(1000)
        ax.plot(xs.numpy(), exact.numpy(), color="black", label="the score")
        ax.plot(xs.numpy(), learned.numpy(), color="C3", linestyle="--", label="$s_\\theta$")
        ax.plot(noisy.numpy(), [-0.9 * top] * 1000, "|", color="C0", alpha=0.3, markersize=6)
        ax.set(xlabel="$x$", xlim=(-6, 6), ylim=(-top, top), title=f"$\\sigma={sigma:.3g}$")
        ax.legend(loc="upper right")
        on_bumps, between = errors(network, sigma)
        print(f"sigma = {sigma:.3g}: s_theta is off by {on_bumps:.3f} on the bumps and by {between:.3f} "
              f"between them, where the score reaches {exact.abs().max().item():.3f}")
    plt.show()


plot_scores(s_theta_one_level, [0.1])

The blue ticks under the plot are 1000 noisy points $\tilde x$. On the bumps, $s_\theta$ follows the score. Between them it is far off: the noisy points almost never fall there, so the loss says nothing about $s_\theta$ there (page 20). And even the exact score did not get the weights right in section 2.

## 4. One network for all the levels (Algorithm 3)

The levels of the deck are ten values of $\sigma$, from $\sigma_1=4$ down to $\sigma_K=0.1$, each $0.66$ times the one before. Algorithm 3 (page 23) picks a level at random at every step and weighs its loss by $\lambda(\sigma)=\sigma^2$ (page 22). We train for 6000 steps on minibatches of $B=1024$ samples; for the last 1500 steps, Adam takes steps ten times shorter, so that the weights settle.

In [ ]:
sigmas = [4.0 * (0.1 / 4.0) ** (k / 9) for k in range(10)]   # sigma_1 = 4 > ... > sigma_K = 0.1
print("the levels:", ", ".join(f"{s:.3g}" for s in sigmas))

s_theta = ScoreNetwork()
optimizer = torch.optim.Adam(s_theta.parameters(), lr=0.001)
B = 1024
for step in range(6000):                                     # Algorithm 3 of page 23
    if step == 4500:
        optimizer.param_groups[0]["lr"] = 0.0001             # the last 1500 steps: shorter steps
    x = data[torch.randint(len(data), (B,))]                   # a minibatch of the data
    sigma = sigmas[torch.randint(len(sigmas), (1,)).item()]    # a level at random
    eps = sigma * torch.randn(B)                               # eps ~ N(0, sigma^2)
    x_tilde = x + eps
    J_hat = sigma ** 2 * ((s_theta(x_tilde, sigma) + eps / sigma ** 2) ** 2).sum() / (2 * B)
    optimizer.zero_grad()
    J_hat.backward()
    optimizer.step()

In [ ]:
plot_scores(s_theta, [sigmas[0], sigmas[len(sigmas) // 3], sigmas[-1]])   # the first level, one further down, the last

At $\sigma_1=4$ the noisy points fill the space between the bumps, and $s_\theta$ follows the score everywhere. At $\sigma_K=0.1$ it follows it on the bumps only, as in section 3. Annealing uses each level where it is good: the large levels move the runs between the bumps, and the small ones shape the bumps.

## 5. Annealed Langevin sampling (Algorithm 4)

`annealed_langevin` is Algorithm 4 of page 25: it starts at $X\sim\mathcal N(0,\sigma_1^2)$ and runs $T_k=100$ steps at each level, with the step length $\Delta_k=\Delta\,\sigma_k^2/\sigma_K^2$ of page 24 and $\Delta=0.005$. For a comparison, as on page 21, the same 2000 starts also run 1000 plain steps at the last level $\sigma_K$ alone. Sampling trains nothing, so it runs under `torch.no_grad()`.

In [ ]:
def annealed_langevin(score, X, Delta=0.005, steps_per_level=100):
    # Algorithm 4 of page 25 for every run at once; returns the path, one row per step, one column per run
    path = [X]
    for sigma in sigmas:
        Delta_k = Delta * sigma ** 2 / sigmas[-1] ** 2          # long steps at high noise
        for t in range(steps_per_level):
            eps = torch.randn(len(X))
            X = X + 0.5 * score(X, sigma) * Delta_k + math.sqrt(Delta_k) * eps
            path.append(X)
    return torch.stack(path)


X0 = sigmas[0] * torch.randn(2000)                           # 2000 runs from N(0, sigma_1^2)
with torch.no_grad():
    plain = langevin_steps(lambda X: s_theta(X, sigmas[-1]), X0, Delta=0.005, steps=1000)
    annealed = annealed_langevin(s_theta, X0)
plot_runs(plain, "only $\\sigma_K$")
plot_runs(annealed, "annealed, from $\\sigma_1$ down to $\\sigma_K$")

The same network and the same starts: at $\sigma_K$ alone, about half the runs end in the left bump, as in section 2; annealed, about 20 %, as in $p^\star$. With the exact score of every $p_\sigma$ in place of $s_\theta$, as on page 21, annealing does as well:

In [ ]:
exact_path = annealed_langevin(exact_score, X0)
print("with the exact score:", " and ".join(f"{s:.1f} %" for s in shares(exact_path[-1])))

## 6. The network as a diffusion model sees it

A diffusion model is trained to predict, from a noisy image, the noise it holds. Page 26 of the deck: the network $\varepsilon_\theta=-\sigma\,s_\theta$ predicts the noise $\varepsilon/\sigma$ that $\tilde x=x+\varepsilon$ holds, scaled to $\mathcal N(0,1)$, and the loss of level $\sigma$ in Algorithm 3 is the loss of a diffusion model:

$$\sigma^2\,\tfrac12\Bigl\|s_\theta(\tilde x,\sigma)+\frac{\varepsilon}{\sigma^2}\Bigr\|^2=\tfrac12\Bigl\|\frac{\varepsilon}{\sigma}-\varepsilon_\theta(\tilde x,\sigma)\Bigr\|^2.$$

The next cell computes both sides on 2000 noisy points at the level $\sigma_3=1.76$.

In [ ]:
def eps_theta(x, sigma):
    # the network as a diffusion model sees it: it predicts the noise eps / sigma that x holds
    return -sigma * s_theta(x, sigma)


sigma = sigmas[2]
x = sample_p_star(2000)
eps = sigma * torch.randn(2000)
x_tilde = x + eps
with torch.no_grad():
    loss_ncsm = sigma ** 2 * ((s_theta(x_tilde, sigma) + eps / sigma ** 2) ** 2).mean() / 2
    loss_noise = ((eps / sigma - eps_theta(x_tilde, sigma)) ** 2).mean() / 2
print(f"NCSM loss times sigma^2: {loss_ncsm.item():.6f}   loss of the predicted noise: {loss_noise.item():.6f}")

The diffusion model removes the predicted noise: $\tilde x-\sigma\,\varepsilon_\theta(\tilde x,\sigma)=\tilde x+\sigma^2 s_\theta(\tilde x,\sigma)$ is its guess of the clean point $x$. The best guess is the average of the clean points that could have given $\tilde x$, and here we can compute it: it is $\tilde x+\sigma^2\,\nabla_x\log p_\sigma(\tilde x)$, the same formula with the exact score. Far from the middle the guess is about a centre; in the middle, where the noise has hidden which bump a point came from, it falls in between.

In [ ]:
#@title Plot the guess against the noisy point (double-click to view the code) { display-mode: "form" }
xs = torch.linspace(-7, 7, 701)
with torch.no_grad():
    guess = xs - sigma * eps_theta(xs, sigma)                # the predicted noise removed
best = xs + sigma ** 2 * exact_score(xs, sigma)              # the average of the clean points that give xs
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(xs.numpy(), best.numpy(), color="black", label="the average clean point")
ax.plot(xs.numpy(), guess.numpy(), color="C3", linestyle="--",
        label="the guess $\\tilde x-\\sigma\\,\\varepsilon_\\theta(\\tilde x,\\sigma)$")
ax.plot(x_tilde.numpy(), [-4.6] * len(x_tilde), "|", color="C0", alpha=0.3, markersize=6)
ax.set(xlabel="the noisy point $\\tilde x$", ylabel="the guess of $x$", xlim=(-7, 7), ylim=(-5, 5),
       title=f"$\\sigma={sigma:.3g}$")
ax.legend(loc="upper left")
plt.show()

Page 27: with $s_\theta=-\varepsilon_\theta/\sigma_k$, the step of Algorithm 4 removes part of the predicted noise and adds fresh noise,

$$X\leftarrow X-\frac{\Delta_k}{2\sigma_k}\,\varepsilon_\theta(X,\sigma_k)+\sqrt{\Delta_k}\,\varepsilon.$$

The predicted noise of $X$ is $\sigma_k\,\varepsilon_\theta(X,\sigma_k)$, so each step removes the part $\Delta_k/(2\sigma_k^2)=\Delta/(2\sigma_K^2)$ of it, the same at every level: a quarter, with $\Delta=0.005$ and $\sigma_K=0.1$. The next cell takes one step both ways, with the same $\varepsilon$.

In [ ]:
Delta = 0.005
sigma_k = sigma
Delta_k = Delta * sigma_k ** 2 / sigmas[-1] ** 2
X = x_tilde
eps = torch.randn(2000)
with torch.no_grad():
    step_with_score = X + 0.5 * s_theta(X, sigma_k) * Delta_k + math.sqrt(Delta_k) * eps            # page 25
    step_with_noise = X - Delta_k / (2 * sigma_k) * eps_theta(X, sigma_k) + math.sqrt(Delta_k) * eps   # page 27
print("the same step:", torch.allclose(step_with_score, step_with_noise, atol=1e-5))
print(f"each step removes {Delta_k / (2 * sigma_k ** 2):.2f} of the predicted noise")

## 7. Try it

* In section 5, give `annealed_langevin` fewer steps, `steps_per_level=10` or `steps_per_level=1`. How few still give the weights of $p^\star$?
* In section 4, train on three levels, `sigmas = [4.0, 0.63, 0.1]`, and run sections 4 and 5 again. Do three levels do as well as ten?
* In section 1, set three bumps, `WEIGHTS = [0.2, 0.3, 0.5]` and `MEANS = [-4.0, 0.0, 4.0]`, and choose Runtime, Run all. Where do the plain steps of section 2 end?